# TAPT (Task-Adaptive Pre-Training) for Korean Movie Review Sentiment Analysis

이 노트북은 kykim/bert-kor-base 모델을 사용하여 영화 리뷰 감정 분석을 위한 TAPT를 구현합니다.

## TAPT란?
- **Task-Adaptive Pre-Training**: 특정 태스크에 맞게 사전 훈련된 모델을 추가로 미세 조정하는 기법
- MLM (Masked Language Modeling)을 통해 도메인 특화된 언어 표현 학습
- 감정 분석 태스크에 특화된 언어 패턴을 학습하여 성능 향상


In [ ]:
# 필요한 라이브러리 설치 및 import
import os
import sys
import re
import warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Transformers 라이브러리
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

# 경고 메시지 무시
warnings.filterwarnings("ignore")

# 시드 설정
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

print("✅ 라이브러리 import 완료")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# 설정 변수들
MODEL_NAME = "kykim/bert-kor-base"
NUM_LABELS = 4
TAPT_EPOCHS = 2
BATCH_SIZE_TRAIN = 16
BATCH_SIZE_EVAL = 32
LEARNING_RATE = 2e-5
TAPT_LEARNING_RATE = 5e-5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128

# 데이터 경로
DATA_PATH = "../../data/raw/train.csv"
TEST_PATH = "../../data/raw/test.csv"

print("📋 설정 완료")
print(f"모델: {MODEL_NAME}")
print(f"TAPT 에포크: {TAPT_EPOCHS}")
print(f"배치 크기: {BATCH_SIZE_TRAIN}")
print(f"최대 길이: {MAX_LENGTH}")


In [ ]:
# 텍스트 전처리 파이프라인 클래스
class TextPreprocessingPipeline:
    """고급 텍스트 전처리 파이프라인"""
    
    def __init__(self):
        self.label_encoder = None
        
    def clean_text(self, text):
        """텍스트 정제"""
        if pd.isna(text):
            return ""
        
        text = str(text)
        
        # HTML 태그 제거
        text = re.sub(r'<[^>]+>', '', text)
        
        # URL 제거
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
        
        # 이메일 제거
        text = re.sub(r'\S+@\S+', '', text)
        
        # 특수문자 정리 (한글, 영문, 숫자, 기본 구두점만 유지)
        text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', '', text)
        
        # 연속된 공백 제거
        text = re.sub(r'\s+', ' ', text)
        
        # 앞뒤 공백 제거
        text = text.strip()
        
        return text
    
    def fit_transform(self, texts, labels=None):
        """훈련 데이터에 맞춰 전처리"""
        print("🔧 텍스트 전처리 시작...")
        
        # 텍스트 정제
        cleaned_texts = [self.clean_text(text) for text in texts]
        
        # 빈 텍스트 제거
        valid_indices = [i for i, text in enumerate(cleaned_texts) if len(text.strip()) > 0]
        cleaned_texts = [cleaned_texts[i] for i in valid_indices]
        
        if labels is not None:
            cleaned_labels = [labels[i] for i in valid_indices]
            print(f"전처리 완료: {len(cleaned_texts):,} 샘플")
            return cleaned_texts, cleaned_labels
        
        print(f"전처리 완료: {len(cleaned_texts):,} 샘플")
        return cleaned_texts
    
    def transform(self, texts):
        """테스트 데이터 전처리"""
        return self.fit_transform(texts)

print("✅ 전처리 파이프라인 클래스 정의 완료")


In [ ]:
# 데이터 로드 및 전처리
print("📁 데이터 로드 중...")
train_df = pd.read_csv(DATA_PATH)

print(f"훈련 데이터: {len(train_df):,} 샘플")
print(f"컬럼: {list(train_df.columns)}")
print(f"라벨 분포:")
print(train_df['label'].value_counts().sort_index())

# 전처리 파이프라인 적용
preprocessor = TextPreprocessingPipeline()
train_texts_processed, train_labels_processed = preprocessor.fit_transform(
    train_df['review'].tolist(), 
    train_df['label'].tolist()
)

# 훈련/검증 데이터 분할
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts_processed, 
    train_labels_processed, 
    test_size=0.2, 
    random_state=RANDOM_SEED,
    stratify=train_labels_processed
)

print(f"\n📊 데이터 분할 완료:")
print(f"훈련 데이터: {len(train_texts):,} 샘플")
print(f"검증 데이터: {len(val_texts):,} 샘플")


In [ ]:
# MLM 데이터셋 클래스 정의
class MLMDataset(Dataset):
    """Masked Language Modeling을 위한 데이터셋"""
    
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        # 토크나이징
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

# 분류를 위한 데이터셋 클래스
class ClassificationDataset(Dataset):
    """감정 분석 분류를 위한 데이터셋"""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        
        # 토크나이징
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("✅ 데이터셋 클래스 정의 완료")


In [ ]:
# 평가 메트릭 함수
def compute_metrics(eval_pred):
    """분류 모델 평가 메트릭 계산"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    
    return {'accuracy': accuracy, 'f1': f1}

# TAPT 수행 함수
def perform_tapt(model, tokenizer, train_texts, tapt_epochs=2):
    """TAPT (Task-Adaptive Pre-Training) 수행"""
    print("🔄 TAPT (Task-Adaptive Pre-Training) 시작...")
    
    # MLM 모델로 변환
    mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
    
    # MLM 데이터셋 생성
    mlm_dataset = MLMDataset(train_texts, tokenizer, MAX_LENGTH)
    
    # TAPT 훈련 설정
    tapt_args = TrainingArguments(
        output_dir="./tapt_model",
        num_train_epochs=tapt_epochs,
        per_device_train_batch_size=BATCH_SIZE_TRAIN,
        learning_rate=TAPT_LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
        seed=RANDOM_SEED,
        fp16=True,
    )
    
    # MLM 데이터 콜레이터
    mlm_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )
    
    # TAPT 트레이너
    tapt_trainer = Trainer(
        model=mlm_model,
        args=tapt_args,
        train_dataset=mlm_dataset,
        data_collator=mlm_collator,
    )
    
    # TAPT 훈련
    print(f"TAPT 훈련 중... ({tapt_epochs} 에포크)")
    tapt_trainer.train()
    
    print("✅ TAPT 완료!")
    return mlm_model

print("✅ TAPT 함수 정의 완료")


In [ ]:
# 토크나이저 및 모델 로드
print("🤖 모델 및 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"토크나이저 로드 완료: {tokenizer.__class__.__name__}")
print(f"어휘 크기: {len(tokenizer):,}")
print(f"특수 토큰: {tokenizer.special_tokens_map}")

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")


In [ ]:
# TAPT 수행
print("🚀 TAPT (Task-Adaptive Pre-Training) 시작")
print("=" * 50)

# TAPT 수행
mlm_model = perform_tapt(
    model=None,  # 모델은 TAPT 함수 내에서 로드
    tokenizer=tokenizer,
    train_texts=train_texts,
    tapt_epochs=TAPT_EPOCHS
)

print("✅ TAPT 완료!")


In [ ]:
# 분류 모델 로드 (TAPT된 가중치 사용)
print("📊 분류 모델 초기화 중...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=NUM_LABELS
)

# TAPT된 가중치를 분류 모델에 적용 (BERT 부분만)
if hasattr(mlm_model, 'bert') and hasattr(model, 'bert'):
    model.bert.load_state_dict(mlm_model.bert.state_dict())
    print("✅ TAPT된 가중치를 분류 모델에 적용 완료")
else:
    print("⚠️ BERT 레이어를 찾을 수 없습니다. 원본 모델을 사용합니다.")

print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"훈련 가능한 파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# 데이터셋 생성
print("📊 데이터셋 생성 중...")
train_dataset = ClassificationDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = ClassificationDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)

print(f"훈련 데이터셋: {len(train_dataset):,} 샘플")
print(f"검증 데이터셋: {len(val_dataset):,} 샘플")

# 데이터 콜레이터
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("✅ 데이터셋 생성 완료")


In [ ]:
# 분류 모델 훈련 설정
training_args = TrainingArguments(
    output_dir="./classification_model",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=BATCH_SIZE_EVAL,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=100,
    eval_steps=500,
    evaluation_strategy="steps",
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    seed=RANDOM_SEED,
    report_to="none",
)

# 트레이너 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ 분류 모델 훈련 설정 완료")


In [ ]:
# 분류 모델 훈련
print("🚀 분류 모델 훈련 시작")
print("=" * 50)

# 훈련 실행
train_result = trainer.train()

print("✅ 분류 모델 훈련 완료!")

# 훈련 결과 출력
print(f"\n📊 훈련 결과:")
print(f"최종 훈련 손실: {train_result.training_loss:.4f}")
print(f"훈련 시간: {train_result.metrics['train_runtime']:.2f}초")
print(f"훈련 샘플 수: {train_result.metrics['train_samples']:,}")


In [ ]:
# 모델 평가
print("📊 모델 평가 중...")

# 검증 데이터로 평가
eval_results = trainer.evaluate()

print(f"\n📈 평가 결과:")
print(f"검증 정확도: {eval_results['eval_accuracy']:.4f}")
print(f"검증 F1 점수: {eval_results['eval_f1']:.4f}")
print(f"검증 손실: {eval_results['eval_loss']:.4f}")

# 상세 분류 리포트
print(f"\n📋 상세 분류 리포트:")
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = val_labels

print(classification_report(y_true, y_pred, 
                          target_names=['매우 부정', '부정', '긍정', '매우 긍정']))


In [ ]:
# 혼동 행렬 시각화
from sklearn.metrics import confusion_matrix

# 혼동 행렬 계산
cm = confusion_matrix(y_true, y_pred)

# 시각화
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['매우 부정', '부정', '긍정', '매우 긍정'],
            yticklabels=['매우 부정', '부정', '긍정', '매우 긍정'])
plt.title('TAPT 모델 혼동 행렬')
plt.xlabel('예측 라벨')
plt.ylabel('실제 라벨')
plt.tight_layout()
plt.show()

print("✅ 시각화 완료")


In [ ]:
# 모델 저장
print("💾 모델 저장 중...")

# 최종 모델 저장
trainer.save_model("./final_tapt_model")
tokenizer.save_pretrained("./final_tapt_model")

print("✅ 모델 저장 완료!")
print("저장 경로: ./final_tapt_model/")

# 모델 요약
print(f"\n📋 모델 요약:")
print(f"기본 모델: {MODEL_NAME}")
print(f"TAPT 에포크: {TAPT_EPOCHS}")
print(f"분류 에포크: 3")
print(f"최종 정확도: {eval_results['eval_accuracy']:.4f}")
print(f"최종 F1 점수: {eval_results['eval_f1']:.4f}")


In [ ]:
# 테스트 데이터 예측 함수
def predict_sentiment(text, model, tokenizer, max_length=128):
    """개별 텍스트에 대한 감정 분석 예측"""
    # 전처리
    preprocessor = TextPreprocessingPipeline()
    cleaned_text = preprocessor.clean_text(text)
    
    # 토크나이징
    inputs = tokenizer(
        cleaned_text,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    
    # 예측
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()
    
    # 라벨 매핑
    label_map = {0: '매우 부정', 1: '부정', 2: '긍정', 3: '매우 긍정'}
    
    return {
        'text': text,
        'cleaned_text': cleaned_text,
        'predicted_label': predicted_class,
        'predicted_sentiment': label_map[predicted_class],
        'confidence': confidence,
        'all_probabilities': predictions[0].tolist()
    }

# 예시 예측
print("🔍 예시 예측 테스트:")
sample_texts = [
    "정말 재미있는 영화였어요! 강력 추천합니다.",
    "별로 재미없고 시간 낭비였습니다.",
    "그냥 그런 영화네요. 특별하지 않아요.",
    "완전 최고예요! 다시 보고 싶어요!"
]

for text in sample_texts:
    result = predict_sentiment(text, model, tokenizer)
    print(f"\n텍스트: {result['text']}")
    print(f"정제된 텍스트: {result['cleaned_text']}")
    print(f"예측 감정: {result['predicted_sentiment']} (신뢰도: {result['confidence']:.3f})")

print("\n✅ TAPT 구현 완료!")
